# PaddleOCR fine-tune: real lines + EDS lines (T4), no Google Drive sign-in

A copy of `ocr_finetune_eds.ipynb` that downloads its three inputs from the team's public Drive folder
instead of mounting a Drive, and keeps its results on the Colab machine for you to download.
The training steps and settings are unchanged. The only code change: `common.py` normally refuses to save a real
run outside `/content/drive`; the first code cell also allows `/content/CVDL-OCR-finetune` (the copy on Colab only).
First run: 25 Nov 2026, run `20261125-123556`, recorded in `evaluation/ocr_training/eds-20261125-123556/`.

Same settings as round 1 (run `20261117-031848`, the current app model): `en_PP-OCRv5_mobile_rec` pretrained weights,
batch 8, learning rate 5e-5 (cosine), at most 20 epochs, seed 0, best epoch by lowest validation crop CER.
`train_paddle.py` also stops early after 3 validations without improvement.

Training data = the round 1 real-only bundle + the EDS line images (366 real + 502 EDS = 868 training lines).
Validation is the round 1 real validation set, unchanged (124 lines). No test data is in either input.

Inputs, downloaded from https://drive.google.com/drive/folders/1YtOoREt07gMIVToH2yD3kUeXTh7FzUGX
(shared as "Anyone with the link"; each file is checked against its SHA-256 before use):
1. `ocr_finetune_bundle.zip`: the round 1 real-only bundle (SHA-256 `7db8ff38…`).
2. `eds_lines.zip`: the EDS line images (SHA-256 `44174ece…`).
3. `ocr_finetune_code-eds.zip`: the `colab/ocr_finetune` code (SHA-256 `13341d1a…`).

How to run:
1. Connect to a Colab **T4 GPU**. In VS Code: Select Kernel > Colab > new server > GPU (T4).
   In the browser: Runtime > Change runtime type > T4 GPU.
2. Run all cells. Nothing asks for Drive access.
3. When training finishes, the last code cell writes `/content/paddleocr-eds-<run>.zip` (the exported
   model in `best/` plus logs, no per-epoch checkpoints). Download it. In VS Code: Colab sidebar > Contents >
   right-click the zip > Download. In the browser: Files panel > right-click > Download.
4. Download before disconnecting: results are only on the Colab machine, so a disconnect loses them.

In [ ]:
# Download the three inputs from the public Drive folder (no sign-in), verify them, add the EDS lines
# to the round 1 bundle, and unpack.
from pathlib import Path
import sys, subprocess, shutil, zipfile, json, time, hashlib, urllib.request

DRIVE_FOLDER = 'https://drive.google.com/drive/folders/1YtOoREt07gMIVToH2yD3kUeXTh7FzUGX'
# Name on Drive: (Drive file ID, expected SHA-256)
FILES = {
    'ocr_finetune_bundle.zip': ('1Cgluz0umVIIfM-jBYoyRT4SSU8pbTZmf',
                                '7db8ff380c56737e979b2d269d1d3eeff1c31d950d2bd886be7558694d80ab46'),
    'eds_lines.zip': ('1svXpaOozysSyMMEvuQScu6FmOZm9dkK3',
                      '44174ece75bd924ace8d183d739b3f684902e0bd0b30b63e20da4bf883999657'),
    'ocr_finetune_code-eds.zip': ('1aM6gDzX4Rq5andXF-pPy216RndFYdDEH',
                                  '13341d1a4122f3042d4b1e77c5b830bc5b4f808b5a1c2f178cee9245b3f559ce'),
}
DOWNLOADS = Path('/content/drive-inputs')
INPUT = Path('/content/CVDL-OCR-inputs')
OUTPUT = Path('/content/CVDL-OCR-finetune')
WEIGHTS = INPUT/'weights'
RUN_ID = time.strftime('%Y%m%d-%H%M%S', time.gmtime())
WORK = Path('/content/ocr-finetune')
for folder in (DOWNLOADS, INPUT, OUTPUT, WORK):
    folder.mkdir(parents=True, exist_ok=True)

def sha(p):
    with Path(p).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def fetch(name):
    file_id, expected = FILES[name]
    path = DOWNLOADS/name
    if not (path.is_file() and sha(path) == expected):
        partial = path.with_name(name + '.part')
        url = f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t'
        with urllib.request.urlopen(url) as response, partial.open('wb') as out:
            shutil.copyfileobj(response, out)
        partial.replace(path)
    actual = sha(path)
    if actual != expected:
        if path.read_bytes()[:1] == b'<':
            raise RuntimeError(f'{name}: Drive sent a web page instead of the file. Check that {DRIVE_FOLDER} '
                               'is still shared as "Anyone with the link".')
        raise RuntimeError(f'{name} on Drive has changed since this notebook was checked '
                           f'(SHA-256 {actual[:8]}…, expected {expected[:8]}…). Ask for the new file before training.')
    print('Verified', name, f'({path.stat().st_size / 1024**2:.1f} MB)')
    return path

ROUND1_BUNDLE = fetch('ocr_finetune_bundle.zip')
EDS_ZIP = fetch('eds_lines.zip')
CODE_ZIP = fetch('ocr_finetune_code-eds.zip')

def extract_safe(archive, target):
    assert archive.is_file(), f'Missing ZIP: {archive}'
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            assert (target/entry.filename).resolve().is_relative_to(target.resolve()), 'Unsafe ZIP path'
        z.extractall(target)
extract_safe(CODE_ZIP, WORK/'code')
SCRIPTS = WORK/'code/colab/ocr_finetune'
# train_paddle.py only saves real runs under /content/drive (Google Drive), so a disconnect can't lose them.
# This notebook saves to the Colab disk and the zip is downloaded at the end, so allow OUTPUT as well.
common = SCRIPTS/'common.py'
guard = "not a.output.is_relative_to(Path('/content/drive'))"
text = common.read_text()
assert text.count(guard) == 1, 'common.py has changed; check its output-folder rule'
common.write_text(text.replace(guard, f"not (a.output.is_relative_to(Path('/content/drive')) or a.output.is_relative_to(Path({str(OUTPUT)!r})))"))
print('Training outputs allowed under', OUTPUT)
MERGED = WORK/f'ocr_finetune_bundle-eds-{RUN_ID}.zip'
subprocess.run([sys.executable, str(SCRIPTS/'add_eds_to_bundle.py'), '--bundle', str(ROUND1_BUNDLE),
                '--eds', str(EDS_ZIP), '--output', str(MERGED)], check=True)
extract_safe(MERGED, WORK/'data')
manifest = json.loads((WORK/'data/bundle_manifest.json').read_text())
print(manifest['counts'])
assert manifest['test_used'] is False and manifest['counts']['test_files'] == 0
assert manifest['counts']['eds_pairs'] > 0

## Install the Paddle training environment (Python 3.12, paddlepaddle-gpu 3.2.0) and PaddleOCR v3.3.2

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
assert shutil.disk_usage('/content').free > 8*1024**3, 'Not enough scratch disk'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.11'], check=True)
uv = shutil.which('uv')
assert uv, 'uv is not on PATH'
TRAINING_PYTHON = '3.12.12'
subprocess.run([uv, 'python', 'install', TRAINING_PYTHON], check=True)
PADDLE_PY = Path('/content/ocr-paddle-py312/bin/python')
if not PADDLE_PY.parent.parent.exists():
    subprocess.run([uv, 'venv', '--seed', '--python', TRAINING_PYTHON, str(PADDLE_PY.parent.parent)], check=True)
def pip(py, *args):
    subprocess.run([str(py), '-m', 'pip', *args], check=True)
pip(PADDLE_PY, 'install', 'paddlepaddle-gpu==3.2.0', '--index-url', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/')

In [ ]:
def checkout(url,tag,commit,dest):
    dest = Path(dest)
    def git(*args):
        result = subprocess.run(['git', *args], text=True, capture_output=True)
        if result.returncode:
            raise RuntimeError('Git failed: ' + ' '.join(args) + '\n' + result.stdout + result.stderr)
        return result.stdout.strip()
    if not (dest/'.git').exists():
        if dest.exists() and (not dest.is_dir() or any(dest.iterdir())):
            raise RuntimeError(f'{dest} exists but is not a Git checkout. Inspect it; it has not been deleted.')
        git('clone','--depth','1','--branch',tag,url,str(dest))
    actual = git('-C',str(dest),'rev-parse','HEAD')
    origin = git('-C',str(dest),'config','--get','remote.origin.url')
    if origin.rstrip('/') != url.rstrip('/'):
        raise RuntimeError(f'{dest} has a different remote: {origin}. No files were overwritten.')
    if actual != commit:
        raise RuntimeError(f'{dest} is at {actual}, expected {commit}. No checkout/reset was forced.')
    changes = git('-C',str(dest),'status','--porcelain','--untracked-files=no')
    if changes:
        raise RuntimeError(f'{dest} has tracked source changes; inspect them before training:\n{changes}')
    print(f'Ready: {dest} at verified {tag} ({actual[:12]})')
checkout('https://github.com/PaddlePaddle/PaddleOCR.git','v3.3.2','95dc316a8490a350c1d1fcf3bc80bf83d4e52450','/content/PaddleOCR')
pip(PADDLE_PY,'install','-r','/content/PaddleOCR/requirements.txt')
pip(PADDLE_PY,'install','numpy>=2.0,<2.3','Pillow==11.3.0','PyYAML==6.0.3','setuptools==75.8.0')  # PaddleOCR deps (scipy/opencv) need numpy 2; paddle 3.2 supports it  # paddle imports setuptools; uv's py3.12 venv lacks it
subprocess.run([str(PADDLE_PY),'-c','import paddle; assert paddle.is_compiled_with_cuda(); paddle.utils.run_check()'],check=True)

## Official Paddle training checkpoint (downloaded to Drive once, hash recorded)

In [ ]:
import urllib.request
PADDLE_INITIAL=WEIGHTS/'paddle-training/en_PP-OCRv5_mobile_rec_pretrained.pdparams'
PADDLE_INITIAL.parent.mkdir(parents=True,exist_ok=True)
origin=PADDLE_INITIAL.with_suffix('.origin.json')
url='https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams'
if not PADDLE_INITIAL.exists():
    assert not origin.exists(), 'Resolve orphaned provenance record first'
    with urllib.request.urlopen(url) as response, PADDLE_INITIAL.open('xb') as output:
        shutil.copyfileobj(response,output)
    with PADDLE_INITIAL.open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
    origin.write_text(json.dumps(dict(url=url,sha256=digest),indent=2)+'\n')
else:
    assert origin.exists(), 'Existing checkpoint needs its original URL/SHA-256 provenance'
    recorded=json.loads(origin.read_text())
    with PADDLE_INITIAL.open('rb') as f:assert hashlib.file_digest(f,'sha256').hexdigest()==recorded['sha256']
    assert recorded['url']==url
print(origin.read_text())

## CPU dry run: 8 pairs, 2 optimiser steps, no saved weights

In [ ]:
def launch(dry=False):
    output = OUTPUT/'paddleocr-eds'/(RUN_ID + ('-dry' if dry else ''))
    args = [str(PADDLE_PY), str(SCRIPTS/'train_paddle.py'), '--data', str(WORK/'data'), '--pretrained', str(PADDLE_INITIAL),
            '--output', str(output), '--seed', '0', '--batch', '8', '--lr', '5e-5', '--epochs', '20',
            '--repo', '/content/PaddleOCR']
    if dry:
        args.append('--dry-run')
    started = time.perf_counter()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end=''); tail = (tail + [line])[-60:]
    if p.wait() != 0:
        raise RuntimeError(f'{"Dry run" if dry else "Training"} failed (exit {p.returncode}). Last output:\n' + ''.join(tail))
    print('dry run' if dry else 'training', 'seconds:', round(time.perf_counter() - started), 'output:', output)
    return output
launch(dry=True)

## Full training on the T4
Watch `epoch_log.csv` in the output folder. Do not change settings mid-run; report errors instead.

In [ ]:
RESULT = launch()

## After training

In [ ]:
print((RESULT/'selection.json').read_text())
print((RESULT/'epoch_log.csv').read_text())
for name in ['inference.json', 'inference.pdiparams', 'inference.yml', 'provenance.json']:
    assert (RESULT/'best'/name).is_file(), f'best/{name} missing'
print('Exported model:', RESULT/'best')

# One zip to download: best/ and the run's small files (logs, config, selection), without the per-epoch checkpoints.
ARCHIVE = Path('/content')/f'paddleocr-eds-{RUN_ID}.zip'
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RESULT.rglob('*')):
        relative = path.relative_to(RESULT)
        if path.is_file() and relative.parts[0] != 'checkpoints':
            z.write(path, Path(RESULT.name)/relative)
    z.write(WORK/'data/bundle_manifest.json', Path(RESULT.name)/'bundle_manifest.json')
print(f'Download {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MB), SHA-256 {sha(ARCHIVE)}')

Download `/content/paddleocr-eds-<run>.zip` (VS Code: Colab sidebar > Contents > right-click > Download;
browser: Files panel > right-click > Download) for evaluation (Step 6). Then Runtime > Disconnect and delete runtime:
everything on the Colab machine, including the results, is deleted.